In [1]:
import imaplib
import email
import pandas as pd
import chardet
import openai
import re
import json
import ftfy
from email.header import decode_header
from datetime import datetime, timedelta

In [ ]:
# Configuración de Gmail
EMAIL = "ki.an.napa.c.o.cha.00@googlemail.com"
PASSWORD = "mail pswd"
IMAP_SERVER = "imap.gmail.com"

In [3]:
# Conectar al servidor IMAP
mail = imaplib.IMAP4_SSL(IMAP_SERVER)
mail.login(EMAIL, PASSWORD)
mail.select("INBOX")  # Carpeta a leer (puedes cambiarla a otra como "Sent")

('OK', [b'6961'])

In [4]:
# Buscar correos (UNSEEN para no leídos, ALL para todos)
status, messages = mail.search(None, "UNSEEN")

In [5]:
# Lista para almacenar datos
emails_data = []

In [6]:
# Leer correos
for num in messages[0].split():
    status, msg_data = mail.fetch(num, "(RFC822)")
    
    for response_part in msg_data:
        if isinstance(response_part, tuple):
            msg = email.message_from_bytes(response_part[1])

            # Obtener información clave
            subject, encoding = decode_header(msg["Subject"])[0]
            subject = subject.decode(encoding or "utf-8") if isinstance(subject, bytes) else subject

            from_email = msg.get("From")
            date = msg.get("Date")

            # Extraer el cuerpo del mensaje
            body = ""
            if msg.is_multipart():
                for part in msg.walk():
                    content_type = part.get_content_type()
                    content_disposition = str(part.get("Content-Disposition"))

                    if content_type == "text/plain" and "attachment" not in content_disposition:
                        try:
                            # Obtener el payload del mensaje
                            payload = part.get_payload(decode=True)

                            # Detectar la codificación
                            detected_encoding = chardet.detect(payload)["encoding"]

                            # Decodificar con la codificación detectada
                            body = payload.decode(detected_encoding, errors="replace")
                        except Exception as e:
                            print(f"Error decodificando el mensaje: {e}")
                            body = part.get_payload()  # Si no se puede decodificar, guardar el payload sin cambios
            else:
                # En caso de que no sea un mensaje multipart, intentamos decodificar el único cuerpo
                payload = msg.get_payload(decode=True)
                detected_encoding = chardet.detect(payload)["encoding"]
                try:
                    body = payload.decode(detected_encoding, errors="replace")
                except Exception as e:
                    print(f"Error decodificando el mensaje: {e}")
                    body = payload.decode("latin-1", errors="replace")  # Alternativa en caso de error

            # Almacenar en lista
            emails_data.append([from_email, subject, date, body])

# Cerrar conexión
mail.logout()

('BYE', [b'LOGOUT Requested'])

In [7]:
# Crear un DataFrame
df = pd.DataFrame(emails_data, columns=["Remitente", "Asunto", "Fecha", "Cuerpo"])

In [8]:
df

,Remitente,Asunto,Fecha,Cuerpo
0,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,RE: Estado cuenta marzo-abril,"Wed, 26 Mar 2025 14:51:50 +0000",Mil disculpas. Se pagan las dos mañana\r\n\r\n...


In [9]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%a, %d %b %Y %H:%M:%S %z').dt.strftime('%d-%m-%Y')

In [10]:
# Función para limpiar el cuerpo del correo
def limpiar_cuerpo(cuerpo):
    #Eliminar texto relacionado con imágenes o contenido no relevante (CID, enlaces, firmas)
    cuerpo_limpio = re.sub(r'\[cid:[^\]]*\]', '', cuerpo)  # Eliminar imágenes incrustadas (cid)
    cuerpo_limpio = re.sub(r'http[s]?://\S+', '', cuerpo_limpio)  # Eliminar enlaces
    cuerpo_limpio = re.sub(r'\n+', ' ', cuerpo_limpio)  # Eliminar saltos de línea múltiples
    cuerpo_limpio = re.sub(r'\s{2,}', ' ', cuerpo_limpio)  # Eliminar espacios múltiples
    #cuerpo_limpio = re.sub(r'(?i)(saludos|atentamente|cordialmente|firma).*', '', cuerpo_limpio)  # Eliminar firma
    cuerpo_limpio = cuerpo_limpio.strip()  # Eliminar espacios al inicio y final

    return cuerpo_limpio

In [11]:
df['Cuerpo'] = df['Cuerpo'].apply(limpiar_cuerpo)

In [12]:
df

,Remitente,Asunto,Fecha,Cuerpo
0,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,RE: Estado cuenta marzo-abril,26-03-2025,Mil disculpas. Se pagan las dos mañana Saludos...


In [13]:
# Configura tu clave de OpenAI
client = openai.Client(api_key="Open AI Api Key")

In [14]:
def extraer_info_correo(fecha, cuerpo):
    """Utiliza OpenAI para extraer datos clave del cuerpo del correo, considerando múltiples facturas."""
    prompt = f"""Analiza el siguiente correo y extrae la información para cada factura mencionada.
    
    - Extrae todas las facturas mencionadas en el texto.
    - Identifica si el remitente confirma el pago (1 si confirma, 0 si no).
    - Extrae la fecha de pago si está mencionada. Si no se menciona, y hay una fecha relativa, como "próximo Viernes", "en una semana", "lunes subsiguiente", calcula la fecha según la recepción del correo
    - Extrae un resumen relevante de la información del correo.
    - Importante si es un hilo de correos, hacer mención a cada uno de los documentos que se mencionan en un inicio, pero manteniendo solo un regitro por documento
    
    **Ejemplo de salida esperada en JSON:**
    ```json
    [
        {{
            "Nro_documento": "123456",
            "Confirma_pago": 1,
            "Fecha_pago": "2024-04-15",
            "Observación": "El pago será realizado el 15 de abril."
        }},
        {{
            "Nro_documento": "654321",
            "Confirma_pago": 0,
            "Fecha_pago": "No especificado",
            "Observación": "Indica que aún no tiene fecha de pago."
        }}
    ]
    ```

    **Fecha del correo:**
    \"\"\"{fecha}\"\"\"
    
    **Texto del correo:**
    \"\"\"{cuerpo}\"\"\"
    
    **IMPORTANTE:** Responde únicamente en formato JSON como en el ejemplo. No incluyas ninguna explicación adicional.
    """

    response = client.chat.completions.create(
        model="gpt-4-turbo",
        messages=[{"role": "system", "content": "Eres un asistente experto en análisis de correos de cobranza."},
                  {"role": "user", "content": prompt}]
    )

    print(f"Respuesta de OpenAI: {response.choices[0].message.content}")  # Depuración

    # Limpiar la respuesta eliminando los delimitadores de código (```json) y (```)
    contenido_respuesta = response.choices[0].message.content.strip("```json").strip("```").strip()

    try:
        resultado = json.loads(contenido_respuesta)  # Convertir respuesta JSON a lista de diccionarios
    except json.JSONDecodeError as e:
        print(f"Error al decodificar JSON: {e}")  # Depuración
        resultado = []

    return resultado

def procesar_dataframe(df):
    datos_extraidos = []
    
    for _, row in df.iterrows():
        print(f"Procesando correo de: {row['Remitente']}")  # Depuración
        if pd.isna(row["Cuerpo"]) or row["Cuerpo"].strip() == "":  # Verificación de datos vacíos
            print(f"Correo vacío o nulo para {row['Remitente']}")  # Depuración
            continue
        
        resultados = extraer_info_correo(row["Fecha"], row["Cuerpo"])  # Extrae info del correo
        
        for resultado in resultados:
            datos_extraidos.append({
                "Deudor": row["Remitente"],
                "Nro_documento": resultado["Nro_documento"],
                "Confirma_pago": resultado["Confirma_pago"],
                "Fecha_pago": resultado["Fecha_pago"],
                "Observación": resultado["Observación"]
            })
    
    # Crear el nuevo DataFrame con los datos extraídos
    if datos_extraidos:
        df_salida = pd.DataFrame(datos_extraidos)
        return df_salida
    else:
        print("No se extrajeron datos.")  # Depuración
        return pd.DataFrame()  # Retorna un DataFrame vacío si no se extrajeron datos

# Asumiendo que tienes el DataFrame 'df' con las columnas "Cuerpo" y "Remitente"
df_salida = procesar_dataframe(df)

if df_salida.empty:
    print("df_salida está vacío.")
else:
    print(df_salida)

Procesando correo de: Juan Pablo Lucero Espinoza <jlucero@eurocapital.cl>
Respuesta de OpenAI: ```json
[
    {
        "Nro_documento": "31",
        "Confirma_pago": 1,
        "Fecha_pago": "2025-03-27",
        "Observación": "Se paga mañana según correo del 26 de marzo."
    },
    {
        "Nro_documento": "32",
        "Confirma_pago": 1,
        "Fecha_pago": "2025-03-27",
        "Observación": "Se paga mañana según correo del 26 de marzo."
    },
    {
        "Nro_documento": "33",
        "Confirma_pago": 1,
        "Fecha_pago": "2025-04-15",
        "Observación": "Pago confirmado para el 15 de abril."
    },
    {
        "Nro_documento": "34",
        "Confirma_pago": 1,
        "Fecha_pago": "2025-04-15",
        "Observación": "Pago confirmado para el 15 de abril."
    },
    {
        "Nro_documento": "35",
        "Confirma_pago": 1,
        "Fecha_pago": "2025-04-15",
        "Observación": "Pago confirmado para el 15 de abril."
    }
]
```
                        

In [15]:
df

,Remitente,Asunto,Fecha,Cuerpo
0,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,RE: Estado cuenta marzo-abril,26-03-2025,Mil disculpas. Se pagan las dos mañana Saludos...


In [16]:
df_salida

,Deudor,Nro_documento,Confirma_pago,Fecha_pago,Observación
0,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,31,1,2025-03-27,Se paga mañana según correo del 26 de marzo.
1,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,32,1,2025-03-27,Se paga mañana según correo del 26 de marzo.
2,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,33,1,2025-04-15,Pago confirmado para el 15 de abril.
3,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,34,1,2025-04-15,Pago confirmado para el 15 de abril.
4,Juan Pablo Lucero Espinoza <jlucero@eurocapita...,35,1,2025-04-15,Pago confirmado para el 15 de abril.


In [17]:
df = df.applymap(lambda x: x.encode('latin-1', 'ignore').decode('latin-1') if isinstance(x, str) else x)

C:\Users\jlucero\AppData\Local\Temp\ipykernel_19880\906070511.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.encode('latin-1', 'ignore').decode('latin-1') if isinstance(x, str) else x)


In [18]:
# Guardar en CSV
df.to_csv("bandeja_entrada.csv", index=False, encoding="latin1", sep=';')

In [19]:
# Guardar en CSV
df_salida.to_csv("resultados_lecura.csv", index=False, encoding="latin1", sep=';')